In [1]:
import torch
from torchvision import transforms
from PIL import Image
import torch.nn.functional as F
import os
import json
import open_clip
from transformers import CLIPProcessor, CLIPVisionModelWithProjection

E:\Program\anaconda3\envs\BCI\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from diffusers import AutoencoderKL, DDPMScheduler, UNet2DConditionModel
unet = UNet2DConditionModel.from_pretrained("./stable-diffusion-v1-5/", subfolder="unet")
unet.requires_grad_(False)

E:\Program\anaconda3\envs\BCI\lib\site-packages\diffusers\utils\outputs.py:63: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


UNet2DConditionModel(
  (conv_in): Conv2d(4, 320, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (time_proj): Timesteps()
  (time_embedding): TimestepEmbedding(
    (linear_1): LoRACompatibleLinear(in_features=320, out_features=1280, bias=True)
    (act): SiLU()
    (linear_2): LoRACompatibleLinear(in_features=1280, out_features=1280, bias=True)
  )
  (down_blocks): ModuleList(
    (0): CrossAttnDownBlock2D(
      (attentions): ModuleList(
        (0-1): 2 x Transformer2DModel(
          (norm): GroupNorm(32, 320, eps=1e-06, affine=True)
          (proj_in): LoRACompatibleConv(320, 320, kernel_size=(1, 1), stride=(1, 1))
          (transformer_blocks): ModuleList(
            (0): BasicTransformerBlock(
              (norm1): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
              (attn1): Attention(
                (to_q): LoRACompatibleLinear(in_features=320, out_features=320, bias=False)
                (to_k): LoRACompatibleLinear(in_features=320, out_features=320

In [3]:
unet.config.block_out_channels

[320, 640, 1280, 1280]

In [4]:
name = 'up_blocks.1.attentions.0.transformer_blocks.0.attn2.processor'
block_id = int(name[len("up_blocks.")])
print(block_id)

1


In [3]:
import torch
import torch.nn.functional as F
import json
import numpy as np
import os
import clip
from utils.Configs import Configs
import re
from tqdm import tqdm
from accelerate import Accelerator
from accelerate.utils import ProjectConfiguration
import itertools
from diffusers import DDPMScheduler, UNet2DConditionModel

from load_eegdatasets import SoloEEGDataset
from torch.utils.data import DataLoader

from model.ReVisModels import ReVisEncoder

# Classifier-Free Guidance Training

def extract_id_from_string(s):
    match = re.search(r'\d+$', s)
    if match:
        return int(match.group())
    return None


# Load the configuration from the JSON file
data_config_path = "data_config.json"
with open(data_config_path, "r") as data_config_file:
    data_config = json.load(data_config_file)

# Access the pretrained SD 1.5
pretrained_model_path = data_config["pretrained_model_path"]
data_path = data_config["data_path"]

default_configs = Configs()

accelerator = Accelerator(
        log_with=default_configs.log_with,
        #project_config=accelerator_project_config,
        gradient_accumulation_steps=default_configs.gradient_accumulation_steps
    )

noise_scheduler = DDPMScheduler.from_pretrained(pretrained_model_path, subfolder="scheduler")
frozen_unet = UNet2DConditionModel.from_pretrained(pretrained_model_path, subfolder="unet")
frozen_unet.requires_grad_(False)
frozen_unet.to(accelerator.device)
model_type = 'ViT-L-14'
sub = 'sub-08'

train_dataset = SoloEEGDataset(data_path, subjects=[sub], train=True)
train_dataloader = DataLoader(train_dataset, batch_size=default_configs.train_batch_size, shuffle=True, num_workers=0,
                              drop_last=True)
eval_dataset = SoloEEGDataset(data_path, subjects=[sub], train=False)
eval_dataloader = DataLoader(eval_dataset, batch_size=default_configs.train_batch_size, shuffle=True,
                                  num_workers=0,drop_last=True)

train_dataloader, eval_dataloader = accelerator.prepare(train_dataloader, eval_dataloader)

eval_loss_sum = 0
with torch.no_grad():
    for idx, batch in enumerate(train_dataloader):
        latents = batch["vae_img_features"]
        noise = torch.randn_like(latents)
        batch_size = latents.shape[0]
        timesteps = torch.randint(0, noise_scheduler.num_train_timesteps, (batch_size,), device=latents.device)
        timesteps = timesteps.long()
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)
        encoder_hidden_states = batch["clip_text_hidden_states"].to(accelerator.device)
        noise_pred = frozen_unet(noisy_latents, timesteps, encoder_hidden_states).sample
        loss = F.mse_loss(noise_pred.float(), noise.float(), reduction="mean")
        eval_avg_loss = accelerator.gather(loss.repeat(batch_size)).mean().item()
        eval_loss_sum += eval_avg_loss
    eval_loss = eval_loss_sum / len(train_dataloader)

print(eval_loss_sum)

eeg_data_tensor torch.Size([16540, 63, 100])
label_tensor torch.Size([16540])
eeg_data_tensor torch.Size([200, 63, 100])
label_tensor torch.Size([200])
325.7065675826743


In [4]:
print(eval_loss)

0.15757453680826045


In [5]:
len(train_dataloader)

2067

In [1]:
import torch
import torch.nn.functional as F
import json
import numpy as np
import os
import clip
from utils.Configs import Configs
import re
from tqdm import tqdm
from accelerate import Accelerator
from accelerate.utils import ProjectConfiguration
import itertools
from diffusers import DDPMScheduler, UNet2DConditionModel

from load_eegdatasets import SoloEEGDataset
from torch.utils.data import DataLoader

from model.ReVisModels import ReVisEncoder

# Classifier-Free Guidance Training

def extract_id_from_string(s):
    match = re.search(r'\d+$', s)
    if match:
        return int(match.group())
    return None


# Load the configuration from the JSON file
data_config_path = "data_config.json"
with open(data_config_path, "r") as data_config_file:
    data_config = json.load(data_config_file)

# Access the pretrained SD 1.5
pretrained_model_path = data_config["pretrained_model_path"]
data_path = data_config["data_path"]

default_configs = Configs()

accelerator = Accelerator(
        log_with=default_configs.log_with,
        #project_config=accelerator_project_config,
        gradient_accumulation_steps=default_configs.gradient_accumulation_steps
    )

noise_scheduler = DDPMScheduler.from_pretrained(pretrained_model_path, subfolder="scheduler")
frozen_unet = UNet2DConditionModel.from_pretrained(pretrained_model_path, subfolder="unet")
frozen_unet.requires_grad_(False)
frozen_unet.to(accelerator.device)
model_type = 'ViT-L-14'
sub = 'sub-08'

# train_dataset = SoloEEGDataset(data_path, subjects=[sub], train=True)
# train_dataloader = DataLoader(train_dataset, batch_size=default_configs.train_batch_size, shuffle=True, num_workers=0,
#                               drop_last=True)
eval_dataset = SoloEEGDataset(data_path, subjects=[sub], train=False)
eval_dataloader = DataLoader(eval_dataset, batch_size=default_configs.train_batch_size, shuffle=True,
                                  num_workers=0,drop_last=True)

eval_dataloader = accelerator.prepare(eval_dataloader)

eval_loss_sum = 0
with torch.no_grad():
    for idx, batch in enumerate(eval_dataloader):
        latents = batch["vae_img_features"]
        noise = torch.randn_like(latents)
        batch_size = latents.shape[0]
        timesteps = torch.randint(0, noise_scheduler.num_train_timesteps, (batch_size,), device=latents.device)
        timesteps = timesteps.long()
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)
        encoder_hidden_states = batch["clip_text_hidden_states"].to(accelerator.device)
        print(f'encoder_hidden_states shape is {encoder_hidden_states.shape}')
        noise_pred = frozen_unet(noisy_latents, timesteps, encoder_hidden_states).sample
        loss = F.mse_loss(noise_pred.float(), noise.float(), reduction="mean")
        eval_avg_loss = accelerator.gather(loss.repeat(batch_size)).mean().item()
        eval_loss_sum += eval_avg_loss
    eval_loss = eval_loss_sum / len(eval_dataloader)

print(eval_loss)

E:\Program\anaconda3\envs\BCI\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
E:\Program\anaconda3\envs\BCI\lib\site-packages\diffusers\utils\outputs.py:63: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
E:\Program\anaconda3\envs\BCI\lib\site-packages\huggingface_hub\file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at openai/clip-vit-large-patch14 were not used when initializing CLIPVisionModelWithProjection: ['text_model.encoder.layers.6.self_attn.q_proj.bias', 'text_m

Some weights of the model checkpoint at openai/clip-vit-large-patch14 were not used when initializing CLIPTextModelWithProjection: ['vision_model.encoder.layers.3.self_attn.q_proj.bias', 'vision_model.encoder.layers.18.self_attn.out_proj.weight', 'vision_model.encoder.layers.20.mlp.fc1.bias', 'vision_model.encoder.layers.1.mlp.fc1.weight', 'vision_model.encoder.layers.22.self_attn.v_proj.bias', 'vision_model.encoder.layers.22.self_attn.q_proj.bias', 'vision_model.encoder.layers.23.self_attn.out_proj.weight', 'vision_model.encoder.layers.12.self_attn.out_proj.bias', 'vision_model.encoder.layers.14.self_attn.q_proj.bias', 'vision_model.encoder.layers.0.layer_norm2.bias', 'vision_model.encoder.layers.15.layer_norm2.bias', 'vision_model.encoder.layers.10.layer_norm2.bias', 'vision_model.encoder.layers.11.mlp.fc2.weight', 'vision_model.encoder.layers.9.mlp.fc1.bias', 'vision_model.encoder.layers.20.self_attn.out_proj.bias', 'vision_model.encoder.layers.21.self_attn.out_proj.weight', 'vision

E:\Program\anaconda3\envs\BCI\lib\site-packages\diffusers\utils\outputs.py:63: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
E:\Program\anaconda3\envs\BCI\lib\site-packages\diffusers\utils\outputs.py:63: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


eeg_data_tensor torch.Size([200, 63, 100])
label_tensor torch.Size([200])


D:\Project\ReVIS\load_eegdatasets.py:92: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  saved_features = torch.load(features_filename)
E:\Program\anaconda3\envs\BCI\lib\site-

encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77

In [8]:
from transformers import CLIPTextModel
text_encoder = CLIPTextModel.from_pretrained(pretrained_model_path, subfolder="text_encoder")
text_encoder.requires_grad_(False)
text_encoder = text_encoder.to(accelerator.device)


def empty_text_encoder(batch_size, accelerator):
    texts = ["" for _ in range(batch_size)]
    text_inputs = torch.cat([clip.tokenize(t) for t in texts]).to(accelerator.device)
    with torch.no_grad():
        text_features = text_encoder(text_inputs)
        text_features = text_features[0]

    #clip_text_hidden_states = F.normalize(text_features, dim=-1).detach()
    return text_features

eval_loss_sum = 0
with torch.no_grad():
    for idx, batch in enumerate(eval_dataloader):
        latents = batch["vae_img_features"]
        noise = torch.randn_like(latents)
        batch_size = latents.shape[0]
        timesteps = torch.randint(0, noise_scheduler.num_train_timesteps, (batch_size,), device=latents.device)
        timesteps = timesteps.long()
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)
        encoder_hidden_states = empty_text_encoder(batch_size, accelerator)
        #encoder_hidden_states = batch["clip_text_hidden_states"].to(accelerator.device)
        print(f'encoder_hidden_states shape is {encoder_hidden_states.shape}')
        noise_pred = frozen_unet(noisy_latents, timesteps, encoder_hidden_states).sample
        loss = F.mse_loss(noise_pred.float(), noise.float(), reduction="mean")
        eval_avg_loss = accelerator.gather(loss.repeat(batch_size)).mean().item()
        eval_loss_sum += eval_avg_loss
    eval_loss = eval_loss_sum / len(eval_dataloader)

print(eval_loss)

encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77

In [9]:
from transformers import CLIPTextModel
text_encoder = CLIPTextModel.from_pretrained(pretrained_model_path, subfolder="text_encoder")
text_encoder.requires_grad_(False)
text_encoder = text_encoder.to(accelerator.device)


def empty_text_encoder(batch_size, accelerator):
    texts = ["" for _ in range(batch_size)]
    text_inputs = torch.cat([clip.tokenize(t) for t in texts]).to(accelerator.device)
    with torch.no_grad():
        text_features = text_encoder(text_inputs)
        text_features = text_features[0]

    clip_text_hidden_states = F.normalize(text_features, dim=-1).detach()
    return clip_text_hidden_states

eval_loss_sum = 0
with torch.no_grad():
    for idx, batch in enumerate(eval_dataloader):
        latents = batch["vae_img_features"]
        noise = torch.randn_like(latents)
        batch_size = latents.shape[0]
        timesteps = torch.randint(0, noise_scheduler.num_train_timesteps, (batch_size,), device=latents.device)
        timesteps = timesteps.long()
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)
        encoder_hidden_states = empty_text_encoder(batch_size, accelerator)
        #encoder_hidden_states = batch["clip_text_hidden_states"].to(accelerator.device)
        print(f'encoder_hidden_states shape is {encoder_hidden_states.shape}')
        noise_pred = frozen_unet(noisy_latents, timesteps, encoder_hidden_states).sample
        loss = F.mse_loss(noise_pred.float(), noise.float(), reduction="mean")
        eval_avg_loss = accelerator.gather(loss.repeat(batch_size)).mean().item()
        eval_loss_sum += eval_avg_loss
    eval_loss = eval_loss_sum / len(eval_dataloader)

print(eval_loss)

encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77, 768])
encoder_hidden_states shape is torch.Size([8, 77

In [2]:
texts = ['' in range(2)]

In [3]:
print(texts.shape)

AttributeError: 'list' object has no attribute 'shape'

In [1]:
import torch
import torch.nn.functional as F
import json
import numpy as np
import os
import clip
from utils.Configs import Configs
import re
from tqdm import tqdm
from accelerate import Accelerator
from accelerate.utils import ProjectConfiguration
import itertools
from diffusers import DDPMScheduler, UNet2DConditionModel

from load_eegdatasets import SoloEEGDataset
from torch.utils.data import DataLoader

from model.ReVisModels import ReVisEncoder

# Classifier-Free Guidance Training

def extract_id_from_string(s):
    match = re.search(r'\d+$', s)
    if match:
        return int(match.group())
    return None


# Load the configuration from the JSON file
data_config_path = "data_config.json"
with open(data_config_path, "r") as data_config_file:
    data_config = json.load(data_config_file)

# Access the pretrained SD 1.5
pretrained_model_path = data_config["pretrained_model_path"]
data_path = data_config["data_path"]

default_configs = Configs()

accelerator = Accelerator(
        log_with=default_configs.log_with,
        #project_config=accelerator_project_config,
        gradient_accumulation_steps=default_configs.gradient_accumulation_steps
    )

noise_scheduler = DDPMScheduler.from_pretrained(pretrained_model_path, subfolder="scheduler")
frozen_unet = UNet2DConditionModel.from_pretrained(pretrained_model_path, subfolder="unet")
frozen_unet.requires_grad_(False)
frozen_unet.to(accelerator.device)
model_type = 'ViT-L-14'
sub = 'sub-08'

train_dataset = SoloEEGDataset(data_path, subjects=[sub], train=True)
train_dataloader = DataLoader(train_dataset, batch_size=default_configs.train_batch_size, shuffle=True, num_workers=0,
                              drop_last=True)
eval_dataset = SoloEEGDataset(data_path, subjects=[sub], train=False)
eval_dataloader = DataLoader(eval_dataset, batch_size=default_configs.train_batch_size, shuffle=True,
                                  num_workers=0,drop_last=True)

train_dataloader, eval_dataloader = accelerator.prepare(train_dataloader, eval_dataloader)

eval_loss_sum = 0
with torch.no_grad():
    for idx, batch in enumerate(train_dataloader):
        latents = batch["vae_img_features"]
        noise = torch.randn_like(latents)
        batch_size = latents.shape[0]
        timesteps = torch.randint(0, noise_scheduler.num_train_timesteps, (batch_size,), device=latents.device)
        timesteps = timesteps.long()
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)
        encoder_hidden_states = batch["clip_text_hidden_states"].to(accelerator.device)
        noise_pred = frozen_unet(noisy_latents, timesteps, encoder_hidden_states).sample
        loss = F.mse_loss(noise_pred.float(), noise.float(), reduction="mean")
        eval_avg_loss = accelerator.gather(loss.repeat(batch_size)).mean().item()
        eval_loss_sum += eval_avg_loss
    eval_loss = eval_loss_sum / len(train_dataloader)

print(eval_loss)

E:\Program\anaconda3\envs\BCI\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
E:\Program\anaconda3\envs\BCI\lib\site-packages\diffusers\utils\outputs.py:63: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
E:\Program\anaconda3\envs\BCI\lib\site-packages\huggingface_hub\file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at openai/clip-vit-large-patch14 were not used when initializing CLIPVisionModelWithProjection: ['text_model.encoder.layers.10.layer_norm1.weight', 'text_mod

Some weights of the model checkpoint at openai/clip-vit-large-patch14 were not used when initializing CLIPTextModelWithProjection: ['vision_model.encoder.layers.20.layer_norm2.weight', 'vision_model.encoder.layers.0.self_attn.k_proj.weight', 'vision_model.encoder.layers.4.mlp.fc1.bias', 'vision_model.encoder.layers.3.layer_norm2.weight', 'vision_model.encoder.layers.22.self_attn.out_proj.weight', 'vision_model.encoder.layers.14.self_attn.q_proj.weight', 'vision_model.encoder.layers.6.self_attn.q_proj.bias', 'vision_model.encoder.layers.3.self_attn.k_proj.weight', 'vision_model.encoder.layers.0.layer_norm2.bias', 'vision_model.encoder.layers.1.self_attn.k_proj.weight', 'vision_model.encoder.layers.19.self_attn.k_proj.bias', 'vision_model.encoder.layers.11.self_attn.v_proj.bias', 'vision_model.encoder.layers.21.self_attn.q_proj.bias', 'vision_model.pre_layrnorm.weight', 'vision_model.encoder.layers.4.self_attn.q_proj.weight', 'vision_model.encoder.layers.9.mlp.fc2.weight', 'vision_model.

E:\Program\anaconda3\envs\BCI\lib\site-packages\diffusers\utils\outputs.py:63: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
E:\Program\anaconda3\envs\BCI\lib\site-packages\diffusers\utils\outputs.py:63: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


eeg_data_tensor torch.Size([16540, 63, 100])
label_tensor torch.Size([16540])


D:\Project\ReVIS\load_eegdatasets.py:92: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  saved_features = torch.load(features_filename)


eeg_data_tensor torch.Size([200, 63, 100])
label_tensor torch.Size([200])


E:\Program\anaconda3\envs\BCI\lib\site-packages\diffusers\configuration_utils.py:135: FutureWarning: Accessing config attribute `num_train_timesteps` directly via 'DDPMScheduler' object attribute is deprecated. Please access 'num_train_timesteps' over 'DDPMScheduler's config object instead, e.g. 'scheduler.config.num_train_timesteps'.
  deprecate("direct config name access", "1.0.0", deprecation_message, standard_warn=False)


0.15303606948989656
